In [1]:
#Clone da repo
!git clone https://github.com/BICLab/Attention-SNN.git
%cd /content/Attention-SNN
#module structure doesnt has __init__.py. therefore adding it
!touch /content/Attention-SNN/MA_SNN/__init__.py
!touch /content/Attention-SNN/MA_SNN/DVSGestures/__init__.py
!touch /content/Attention-SNN/MA_SNN/DVSGestures/CNN/__init__.py
#numpy.int,float are are not there anymore in ts pytorch ver nowadays
!find /content/Attention-SNN/ -name "*.py" -exec sed -i "s/np.int/int/g" {} +
!find /content/Attention-SNN/ -name "*.py" -exec sed -i "s/np.float/float/g" {} +
#again pytorch doesnt have verbose
target_net = "/content/Attention-SNN/MA_SNN/DVSGestures/CNN/Networks/Att_SNN.py"
!sed -i 's/verbose=config.lr_scheduler_verbose//g' {target_net}
!sed -i 's/, )/)/g' {target_net}
#hard patching gpu batch size because other it runs outta memory
config_path = "/content/Attention-SNN/MA_SNN/DVSGestures/CNN/Config.py"
!sed -i "s/self.device = 'cpu'/self.device = 'cuda'/g" {config_path}
!sed -i "s/self.device_ids = range(0, 0)/self.device_ids = range(0, 1)/g" {config_path}
!sed -i "s/self.batch_size = 128/self.batch_size = 8/g" {config_path}
!sed -i "s/self.batch_size_test = 128/self.batch_size_test = 8/g" {config_path}
#Bypass authors faltu safety checks and change cpu to gpu everywhere
!sed -i 's/torch.device("cuda" if torch.cuda.is_available() else "cpu")/"cuda"/g' /content/Attention-SNN/MA_SNN/DVSGestures/CNN/Att_SNN.py
!sed -i "s/torch.cuda.device_count()/1/g" /content/Attention-SNN/MA_SNN/DVSGestures/CNN/Att_SNN.py

print("Env patched")

fatal: destination path 'Attention-SNN' already exists and is not an empty directory.
/content/Attention-SNN
Env patched


In [2]:
import os
%cd /content/Attention-SNN/MA_SNN/DVSGestures/data/

#Download raw dataset
if not os.path.exists('DvsGesture.tar.gz'):
    !wget -O DvsGesture.tar.gz "https://www.dropbox.com/s/cct5kyilhtsliup/DvsGesture.tar.gz?dl=1"
    !tar -xvf DvsGesture.tar.gz

#preprocessing script to create HDF5 data

!python DVS_Gesture.py
print("dun.should work now hopefully")

/content/Attention-SNN/MA_SNN/DVSGestures/data
DVS-Gestures
/content/Attention-SNN/MA_SNN/DVSGestures/data/DVS_Gesture.py:21: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(path=dirs)
processing train data...
1
0
1
2
3
4
5
6
7
8
9
10
11
2
0
1
2
3
4
5
6
7
8
9
10
11
3
0
1
2
3
4
5
6
7
8
9
10
11
4
0
1
2
3
4
5
6
7
8
9
10
11
5
0
1
2
3
4
5
6
7
8
9
10
11
6
0
1
2
3
4
5
6
7
8
9
10
11
7
0
1
2
3
4
5
6
7
8
9
10
8
0
1
2
3
4
5
6
7
8
9
10
11
9
0
1
2
3
4
5
6
7
8
9
10
11
10
0
1
2
3
4
5
6
7
8
9
10
11
11
0
1
2
3
4
5
6
7
8
9
10
11
12
0
1
2
3
4
5
6
7
8
9
10
11
13
0
1
2
3
4
5
6
7
8
9
10
11
14
0
1
2
3
4
5
6
7
8
9
10
11
15
0
1
2
3
4
5
6
7
8
9
10
11
16
0
1
2
3
4
5
6
7
8
9
10
11
17
0
1
2
3
4
5
6
7
8
9
10
11
18
0
1
2
3
4
5
6
7
8
9
10
11
19
0
1
2
3
4
5
6
7
8
9
10
11
20
0
1
2
3
4
5
6
7
8
9
10
11
21
0
1
2
3
4
5
6
7
8
9
10
11
22
0
1
2
3
4
5
6
7
8
9
10
11
23
0
1
2
3
4
5
6
7
8
9
1

In [7]:
#pathing some files where it coulnd find
network_file = "/content/Attention-SNN/MA_SNN/DVSGestures/CNN/Networks/Att_SNN.py"
config_file = "/content/Attention-SNN/MA_SNN/DVSGestures/CNN/Config.py"

#removing verbose from schdule caller and removing commas and stuff
!sed -i 's/verbose=[A-Za-z]*//g' {network_file}
!sed -i 's/, ,/,/g' {network_file}
!sed -i 's/, )/)/g' {network_file}

#Forcing numwork to  get to only 2 workers
!sed -i "s/self.num_work = 8/self.num_work = 2/g" {config_file}

print("complete")

complete


ok should be ready to launch p much

In [ ]:
import sys
import os
import torch

#see if path is still there
project_root = "/content/Attention-SNN/MA_SNN"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

#the main launch part
%cd /content/Attention-SNN/MA_SNN/DVSGestures
from DVSGestures.CNN import Att_SNN

print(f"starting on {torch.cuda.get_device_name(0)}")
Att_SNN.main()

/content/Attention-SNN/MA_SNN/DVSGestures
starting on Tesla T4
cuda
range(0, 1)
dt==25
T==60
attention==no
c_ratio==8
t_ratio==5
epoch==0
num_epochs==300
onlyTest==False
pretrained_path==None
batch_size==8
batch_size_test==8
init_method==None
ds==4
in_channels==2
im_width==32
im_height==32
target_size==11
clip==10
is_train_Enhanced==True
is_spike==False
interval_scaling==False
beta==0
alpha==0.3
Vreset==0
Vthres==0.3
reduction==16
T_extend_Conv==False
T_extend_BN==False
h_conv==False
mem_act==<built-in method relu of type object at 0x7eba75ee4b40>
mode_select==spike
TR_model==NTR
track_running_stats==True
a==0.5
lens==0.25
lr==0.0001
betas==[0.9, 0.999]
eps==1e-08
weight_decay==0
lr_scheduler==True
lr_scheduler_epoch==25
name==no_SNN(CNN)-DVS-Gesture_dt=25ms_T=60
modelPath==/content/Attention-SNN/MA_SNN/DVSGestures/CNN/Result
modelNames==no_SNN(CNN)-DVS-Gesture_dt=25ms_T=60.t7
recordPath==/content/Attention-SNN/MA_SNN/DVSGestures/CNN/Result
recordNames==no_SNN(CNN)-DVS-Gesture_dt=25ms_

Train:Epoch[1/300]: 100%|██████████| 147/147 [01:50<00:00,  1.33it/s, Loss=0.119]


epoch: 1
dt: 25
T: 60
Tarin loss:0.13619
Train acc: 50.298


Test:Epoch[1/300]:  42%|████▏     | 15/36 [00:30<00:37,  1.80s/it, Loss=0.13] /content/Attention-SNN/MA_SNN/DVSGestures/DVS_gesture_data_process/DVS_Gesture_dataloders.py:174: RuntimeWarning: overflow encountered in scalar subtract
  if clip * T * dt - (end_time - start_time) > 0:
/content/Attention-SNN/MA_SNN/DVSGestures/DVS_gesture_data_process/DVS_Gesture_dataloders.py:176: RuntimeWarning: overflow encountered in scalar subtract
  np.floor((clip * T * dt - (end_time - start_time)) / clip))
Test:Epoch[1/300]: 100%|██████████| 36/36 [01:09<00:00,  1.93s/it, Loss=0.14]


Test loss:0.01486
Test acc: 49.653
Saving..
beat acc: 49.65277777777778


...

Train:Epoch[35/300]: 100%|██████████| 147/147 [01:46<00:00,  1.39it/s, Loss=0.0232]


epoch: 35
dt: 25
T: 60
Tarin loss:0.02046
Train acc: 96.255


Test:Epoch[35/300]:  42%|████▏     | 15/36 [00:30<00:40,  1.92s/it, Loss=0.0109]/content/Attention-SNN/MA_SNN/DVSGestures/DVS_gesture_data_process/DVS_Gesture_dataloders.py:174: RuntimeWarning: overflow encountered in scalar subtract
  if clip * T * dt - (end_time - start_time) > 0:
/content/Attention-SNN/MA_SNN/DVSGestures/DVS_gesture_data_process/DVS_Gesture_dataloders.py:176: RuntimeWarning: overflow encountered in scalar subtract
  np.floor((clip * T * dt - (end_time - start_time)) / clip))
Test:Epoch[35/300]: 100%|██████████| 36/36 [01:08<00:00,  1.89s/it, Loss=0.0279]


Test loss:0.00298
Test acc: 87.500
beat acc: 90.27777777777777


Train:Epoch[36/300]:  93%|█████████▎| 136/147 [01:37<00:08,  1.28it/s, Loss=0.0204]